# Model SIM 

Model SIM is introduced in chapter 3 of {cite:t}`GodleyLavoie2006MonetaryEconomicsIntegrated` "Monetary Economics: An Integrated Approach to Credit, Money, Income, Production and Wealth" and represents the "simplest model with government money". That is, this is a model with only _outside money_ (from the government).

## Module Contents

As with all `MacroStat` models, SIM is divided into Variables, Parameters (fixed constants), Scenarios, and the Behavior (model initialization and steps). The module-level documentation, such as all variables/parameters/scenarios and their notation or the behavioral equations associated with each function of `BehaviorSIM.py` can be seen in:

```{eval-rst}
.. toctree::
    :maxdepth: 2

    Variables <GL06SIM/variables.rst>
    Parameters <GL06SIM/parameters.rst>
    Equations <GL06SIM/behavior.rst>
    Scenarios <GL06SIM/scenarios.rst>
```

:::{Note}
The remainder of this page gives an introduction to the model, notes on how it is implemented in `MacroStat` and then shows some of the model dynamics by replicating the relevant graphs of Godley and Lavoie (2006).
:::

## Model Overview

### Behavioral Equations

The SIM model is introduced in Chapter 3, and consists of the following 11 equations and 11 unknowns:

1. Consumption good supply equals demand
```{math}
:label: gl06_sim_eq301_consumptionClearing
C_s(t) = C_d(t)
```
2. Governmend good supply equals demand
```{math}
:label: gl06_sim_eq302_governmentClearing
G_s(t) = G_d(t)
```
3. Tax supply equals demand
```{math}
:label: gl06_sim_eq303_taxClearing
T_s(t) = T_d(t)
```
4. Labour supply equals demand
```{math}
:label: gl06_sim_eq304_labourClearing
N_s(t) = N_d(t)
```
5. Disposable income is wage income minus taxes
```{math}
:label: gl06_sim_eq305_disposableIncome
YD(t) = W(t)\cdot N_s(t) - T_s(t)
```
6. Tax demand is a fixed proportion of wage income
```{math}
:label: gl06_sim_eq306_taxDemand
T_d(t) = \theta\cdot W(t)\cdot N_s(t)
```
7. Consumption demand is a share of disposable income and deposits
```{math}
:label: gl06_sim_eq307_consumptionDemand
C_d(t) = \alpha_1\cdot YD(t) + \alpha_2\cdot H_h(t-1)
```
8. The change in government stock of money is demand minus tax
```{math}
:label: gl06_sim_eq308_governmentDeposits
\Delta H_s(t) = H_s(t) - H_s(t-1) = G_d(t) - T_d(t)
```
9. The change in household deposits is disposable income minus expenditure
```{math}
:label: gl06_sim_eq309_householdDeposits
\Delta H_h(t) = H_h(t) - H_h(t-1) = YD(t) - C_d(t)
```
8. Total national income is consumption of households and government
```{math}
:label: gl06_sim_eq310_nationalIncome
Y(t) = C_s(t) + G_s(t)
```
8. Labour Demand is national income over wages
```{math}
:label: gl06_sim_eq311_labourDemand
N_d(t) = \frac{Y(t)}{W(t)}
```

With the redundant equation being
```{math}
:label: gl06_sim_eq312_redundant
\Delta H_s(t) = \Delta H_h(t)
```


### Transaction Flow Matrix

The accounting of transactions for model SIM is as follows:

:::{table} Accounting Transaction Matrix for Model SIM
:widths: auto
:align: center

|                          | Households       | Production    | Government       | $\Sigma$ |
| :----------------------- | :--------------: | :-----------: | :--------------: | :------: |
| Consumption              | $-C(t)$          | $+C(t)$       |                  | 0        |
| Govt. Expenditure        |                  | $+G(t)$       | $-G(t)$          | 0        |
| [Output]                 |                  |               |                  | 0        |
| Factor Income (Wage)     | $+W(t)N_s(t)$    | $+W(t)N_s(t)$ |                  | 0        |
| Taxes                    | $-T_s(t)$        |               | $+T_s(t)$        | 0        |
| Change in Stock of Money | $-\Delta H_s(t)$ |               | $+\Delta H_d(t)$ | 0        |
| $\Sigma$                 | 0                | 0             | 0                | 0        |
:::

### Balance Sheet Matrix

:::{table} Accounting Transaction Matrix for Model SIM
:widths: auto
:align: center

|                          | Households | Production | Government | $\Sigma$ |
| :----------------------- | :--------: | :--------: | :--------: | :------: |
| Money Stock              | $+H_h(t)$  | 0          | $-H_s(t)$  | 0        |
:::

## Implementation in MacroStat

Transposing these eleven equations to the `MacroStat` framework, we consider that there are:

1. Three parameters (fixed constants): $\alpha_1$, $\alpha_2$, and $\theta$ (see [Parameters](GL06SIM/parameters.rst))
2. Two scenario variables : $G_d(t)$ and $W(t)$ (see [Scenarios](GL06SIM/scenarios.rst))
3. The remaining 14 tracked series are variables (see [Variables](GL06SIM/variables.rst))

### Behavioral Modeling

For the implementation of the behavioral equations (see [Behavior](GL06SIM/behavior.rst)), most prior implementations have made use of some form of linear solver or iteration until the system is solved. To simplify the implementation in Macrostat, we can note that the system can be solved analytically for a given timestep as follows:

Substitute into Eq. {eq}`gl06_sim_eq311_labourDemand` to obtain
```{math}
N_d(t) =\frac{1}{W(t)}(C_d(t) + G_d(t))
```
where we can already note that $W(t)$ and $G_d(t)$ are exogenously given (they are scenario variables). This leaves us to solve for $C_d(t)$, where we can use Eq. {eq}`gl06_sim_eq307_consumptionDemand` and Eq. {eq}`gl06_sim_eq305_disposableIncome` to obtain
```{math}
N_d(t) =\frac{1}{W(t)}(\alpha_1(1-\theta)W(t)N_s(t) + \alpha_2 H_h(t-1) + G_d)
```
now noting from Eq. {eq}`gl06_sim_eq304_labourClearing` that supply=demand, we can rewrite the above as
```{math}
:label: gl06_sim_eq13_solutionLabourDemand
N_d(t) =\frac{\alpha_2 H_h(t-1) + G_d}{W(t)(1-\alpha_1(1-\theta))}
```

Therefore, for a given period $t$ we can solve the system by solving, in order:
1. Eq. {eq}`gl06_sim_eq302_governmentClearing` given the scenario variable
2. Eq. {eq}`gl06_sim_eq13_solutionLabourDemand` for labour demand based on prior information $H_h(t)$ and scenario variables $W(t)$ and $G_d(t)$ (this replaces the need to run Eq. {eq}`gl06_sim_eq311_labourDemand`)
3. Eq. {eq}`gl06_sim_eq304_labourClearing`
4. Tax demand, given labour, Eq. {eq}`gl06_sim_eq306_taxDemand`
5. Tax supply, given demand, Eq. {eq}`gl06_sim_eq303_taxClearing`
6. Disposable income, given labour supply and tax demand, Eq. {eq}`gl06_sim_eq305_disposableIncome`
7. Consumption demand, given disposable income, Eq. {eq}`gl06_sim_eq307_consumptionDemand`
8. Consumption supply, given demand, Eq. {eq}`gl06_sim_eq301_consumptionClearing`
9. Government stock of money, Eq. {eq}`gl06_sim_eq308_governmentDeposits`
10. Household stock of deposits, Eq. {eq}`gl06_sim_eq309_householdDeposits`
11. National income, Eq. {eq}`gl06_sim_eq310_nationalIncome`

This is implemented as such in the [Behavior](GL06SIM/behavior.rst) class. 

## Model Dynamics

### Preparatory Steps

In [1]:
%load_ext autoreload
%autoreload 2

In [67]:
import macrostat
from macrostat.models import GL06SIM, get_available_models, get_model, get_model_classes

In [100]:
modelClass = get_model_classes("GL06SIM")
model = modelClass.Model()
model.simulate()

In [101]:
model.variables.to_pandas()

,ConsumptionDemand,ConsumptionSupply,GovernmentDemand,GovernmentSupply,TaxSupply,TaxDemand,LabourSupply,LabourDemand,DisposableIncome,Wages,GovernmentMoneyStock,HouseholdMoneyStock,NationalIncome
,0,0,0,0,0,0,0,0,0,0,0,0,0
0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,79.999962,79.999962,0.0,20.0,19.999992,19.999992,99.999962,99.999962,79.999969,0.0,79.999878,79.999947,99.999962
96,79.999962,79.999962,0.0,20.0,19.999992,19.999992,99.999962,99.999962,79.999969,0.0,79.999886,79.999947,99.999962
97,79.999962,79.999962,0.0,20.0,19.999992,19.999992,99.999962,99.999962,79.999969,0.0,79.999893,79.999947,99.999962
